# Court-Sight Pipeline Runner

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your match video to Google Drive at `MyDrive/court-sight/video.mp4`
3. Run all cells in order (Runtime → Run all)

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/court-sight'
VIDEO_PATH = f'{DRIVE_DIR}/video.mp4'
OUT_DIR    = f'{DRIVE_DIR}/output'

import os
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(VIDEO_PATH), f'Video not found at {VIDEO_PATH} — upload it to Drive first'
print('Drive mounted, video found.')

In [ ]:
# ── Cell 2: Clone repo and install dependencies ──────────────────────────────
# Clone the branch
!git clone --branch claude/tennis-shot-feedback-Tclpt --depth 1 \
    https://github.com/azhou555/court-sight.git /content/court-sight

%cd /content/court-sight

# Install only what Colab doesn't already have
# (torch, torchvision, numpy, opencv, pandas, scipy are pre-installed)
!pip install -q \
    ultralytics>=8.4 \
    boxmot>=10.0 \
    pyyaml \
    gdown>=5.1 \
    xgboost>=2.0 \
    lightgbm>=4.3 \
    scikit-learn>=1.4

print('Dependencies installed.')

In [ ]:
# ── Cell 3: Copy MCP data from repo into Drive output dir ───────────────────
import shutil

MCP_PATH = f'{DRIVE_DIR}/mcp_set3.csv'

# If not already in Drive, copy from the repo
if not os.path.exists(MCP_PATH):
    repo_mcp = 'data/external/matches/cincinnati_2023_final/mcp_set3.csv'
    shutil.copy(repo_mcp, MCP_PATH)
    print(f'Copied MCP data to {MCP_PATH}')
else:
    print('MCP data already in Drive.')

In [ ]:
# ── Cell 4: Verify GPU ───────────────────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 5: Run the pipeline ─────────────────────────────────────────────────
import subprocess, sys, time, json

PROGRESS_FILE = f'{OUT_DIR}/progress.txt'

cmd = [
    sys.executable, '-u', '-m', 'scripts.process_match',
    '--video',  VIDEO_PATH,
    '--mcp',    MCP_PATH,
    '--out',    OUT_DIR,
    '--device', device,
]

print('Starting pipeline...\n')
t0 = time.time()

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)

with open(PROGRESS_FILE, 'w') as pf:
    for line in proc.stdout:
        print(line, end='')          # show in Colab cell
        pf.write(line)               # write to Drive for remote monitoring
        pf.flush()

proc.wait()
elapsed = time.time() - t0

status = 'SUCCESS' if proc.returncode == 0 else f'FAILED (exit {proc.returncode})'
summary_line = f'\nPipeline {status} in {elapsed:.0f}s\n'
print(summary_line)

with open(PROGRESS_FILE, 'a') as pf:
    pf.write(summary_line)

# Print final summary if it exists
summary_path = f'{OUT_DIR}/summary.json'
if os.path.exists(summary_path):
    print(json.dumps(json.load(open(summary_path)), indent=2))